# 01. Детекция текста

| вход | выход |
|---|---|
| файл чертежа | маска текстовой области `text_region_mask` |

Текст ищется не ради текста, а чтобы буквы не участвовали в оценке границ классов толщины на
следующих этапах. Поэтому маска может быть грубой: мы ничего не стираем, а только не учитываем.

Весь расчет живет в пакете `draftslice`, блокнот только подбирает параметры и смотрит картинки.
Результат кладется в кеш и оттуда читается блокнотом 02.

In [3]:
from pathlib import Path

from artifact_cache import ArtifactCache
from ipywidgets import Dropdown, FloatSlider, interactive
from visualization import (
    draw_text_candidates,
    draw_text_region,
    show_bgr_image,
    show_probability_maps,
    show_probability_summary,
)

from draftslice.common_types import Mask, Mat
from draftslice.image_preparation import load_bgr_image
from draftslice.text_detection import (
    TextCandidates,
    TextKernels,
    TextProbabilityDetector,
    accept_candidates,
    build_text_region_mask,
    find_text_candidates,
    take_accepted_kernels,
)

DATASET_FOLDER = Path("/mnt/data/datasets/Сравнение чертежей/Эскизы датасет")
MODEL_PATH = Path("../models/PP-OCRv5_server_det_infer.onnx")
DETECTOR_SCALES: tuple[int, ...] = (320, 640, 2000)

ModuleNotFoundError: No module named 'artifact_cache'

## Выбор файла

In [ ]:
source_files: list[Path] = sorted(DATASET_FOLDER.glob("*.jpg"))
print(f"файлов в датасете: {len(source_files)}")


def run_source(file_index: int) -> Path:
    """Выбор файла из датасета."""
    source_path = source_files[file_index]
    image = load_bgr_image(source_path)
    show_bgr_image(image, f"[{file_index}] {source_path.stem} | {image.shape[1]}x{image.shape[0]}")
    return source_path


ui_source = interactive(
    run_source,
    file_index=Dropdown(
        options=[(f"[{index}] {path.stem}", index) for index, path in enumerate(source_files)],
        value=0,
        description="файл",
    ),
)
ui_source

In [ ]:
source_path: Path = ui_source.result
image: Mat = load_bgr_image(source_path)
print(f"кадр {image.shape[1]}x{image.shape[0]} | {image.dtype}")

## Карта вероятностей

Сеть прогоняется на нескольких масштабах: мелкие надписи различимы на крупном масштабе, крупные
заголовки на мелком. Карты усредняются.

In [ ]:
detector = TextProbabilityDetector(MODEL_PATH)
probability_maps = detector.detect_multiscale_maps(image, DETECTOR_SCALES)

for long_side, probability_map in zip(DETECTOR_SCALES, probability_maps, strict=True):
    print(f"масштаб {long_side}: max {probability_map.max():.2f}")
show_probability_maps(probability_maps, DETECTOR_SCALES)

In [ ]:
text_probability = probability_maps.mean(axis=0)
print(f"карта {text_probability.shape} | min {text_probability.min():.3f} max {text_probability.max():.3f}")
show_probability_summary(text_probability)

## Отбор ядер

Логика DBPostProcess: контуры бинарной карты, минимальный прямоугольник, средняя вероятность внутри
него. Порог `binary_threshold` отвечает только за связность, решение принимает `box_threshold`.

In [ ]:
def report_candidates(candidates: TextCandidates, accepted: Mask) -> None:
    """Сколько кандидатов принято и чем они отличаются."""
    print(f"кандидатов {len(candidates)} | принято {int(accepted.sum())}")
    if len(candidates):
        print(
            f"score: принятые медиана {candidates.score[accepted].mean():.2f} | "
            f"отклоненные медиана {candidates.score[~accepted].mean():.2f}"
        )


def run_candidates(binary_threshold: float, box_threshold: float, minimum_short_side: float) -> TextKernels:
    """Слайдеры отбора текстовых ядер."""
    candidates = find_text_candidates(text_probability, binary_threshold)
    accepted = accept_candidates(candidates, box_threshold, minimum_short_side)
    report_candidates(candidates, accepted)
    draw_text_candidates(image, candidates, accepted, f"ядра: binary {binary_threshold:.2f} | box {box_threshold:.2f}")
    return take_accepted_kernels(candidates, accepted)


ui_candidates = interactive(
    run_candidates,
    binary_threshold=FloatSlider(
        value=0.30, min=0.05, max=0.90, step=0.05, continuous_update=False, description="binary"
    ),
    box_threshold=FloatSlider(value=0.40, min=0.10, max=0.95, step=0.05, continuous_update=False, description="box"),
    minimum_short_side=FloatSlider(
        value=3.0, min=0.0, max=20.0, step=1.0, continuous_update=False, description="min сторона"
    ),
)
ui_candidates

## Раздувание ядер

Контур принятого ядра раздувается на расстояние, пропорциональное отношению площади к периметру.
Берем с запасом: лишний захват линии стоит дешево, пропущенный текст дорого.

In [ ]:
def run_text_region(unclip_ratio: float) -> Mask:
    """Слайдер раздувания ядер."""
    kernels = ui_candidates.result
    region = build_text_region_mask(kernels, unclip_ratio, image.shape[:2])
    print(f"ядер {len(kernels)} | область {region.mean() * 100:.3f}% пикселей кадра")
    draw_text_region(image, region, f"текстовая область: unclip {unclip_ratio:.1f}")
    return region


ui_text_region = interactive(
    run_text_region,
    unclip_ratio=FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, continuous_update=False, description="unclip"),
)
ui_text_region

## Артефакт для следующего этапа

In [ ]:
text_region_mask: Mask = ui_text_region.result
stage_parameters = {
    "detector_scales": list(DETECTOR_SCALES),
    **ui_candidates.kwargs,
    **ui_text_region.kwargs,
}

cache = ArtifactCache()
saved_path = cache.save(source_path, "text_region", stage_parameters, text_region_mask=text_region_mask)
print(f"сохранено: {saved_path}")
print(f"параметры: {stage_parameters}")